# 📰 Fake News Detection with Machine Learning
**Step-by-step Tutorial on Text Preprocessing, TF-IDF Vectorization, Model Training, Evaluation, and Feature Explainability.**

---
### 📌 Pipeline Steps:
1. **Load Data**: Loading `True.csv` and `Fake.csv` from the benchmark ISOT dataset.
2. **Text Cleaning**: Lowercase, regex URL/HTML stripping, punctuation & digit removal.
3. **Train-Test Split & TF-IDF**: Numerical transformation with unigrams and bigrams.
4. **Model Training**: Fitting an L2-regularized Logistic Regression classifier.
5. **Evaluation**: Classification report and Confusion Matrix heatmap.
6. **Explainability**: Inspecting vocabulary coefficients that drive Fake vs. Real classifications.

## 1. Setup & Imports

In [ ]:
import os
import re
import string
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.pipeline import Pipeline

plt.style.use('seaborn-v0_8-whitegrid')
print('Dependencies loaded successfully!')

## 2. Load & Inspect Dataset

In [ ]:
# Load sample from True and Fake news datasets
df_true = pd.read_csv('data/True.csv', nrows=5000)
df_fake = pd.read_csv('data/Fake.csv', nrows=5000)

df_true['label'] = 1  # 1 = Real News
df_fake['label'] = 0  # 0 = Fake News

df = pd.concat([df_true, df_fake], ignore_index=True)
df = df.sample(frac=1.0, random_state=42).reset_index(drop=True)

print(f'Total sample records: {len(df)}')
print('\nClass distribution:')
print(df['label'].value_counts())
df.head(3)

## 3. Text Preprocessing

In [ ]:
def clean_text(text):
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    text = re.sub(r'<.*?>', ' ', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\d+', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

# Combine headline and article body
df['content'] = (df['title'].fillna('') + ' ' + df['text'].fillna('')).apply(clean_text)

print('Cleaned sample:')
print(df['content'].iloc[0][:140])

## 4. Train-Test Split & TF-IDF Vectorization

In [ ]:
X = df['content']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=50000,
        min_df=2,
        ngram_range=(1, 2),
        stop_words='english'
    )),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

pipeline.fit(X_train, y_train)
print('Pipeline trained successfully!')

## 5. Model Evaluation & Performance Metrics

In [ ]:
y_pred = pipeline.predict(X_test)
print(f'Test Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%\n')
print('Classification Report:')
print(classification_report(y_test, y_pred, target_names=['Fake (0)', 'Real (1)']))

# Plot Confusion Matrix Heatmap
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Fake', 'Real'], yticklabels=['Fake', 'Real'])
plt.title('Confusion Matrix')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.tight_layout()
plt.show()

## 6. Model Explainability (Feature Attribution)

In [ ]:
tfidf = pipeline.named_steps['tfidf']
clf = pipeline.named_steps['classifier']
vocab = np.array(tfidf.get_feature_names_out())
coefs = clf.coef_[0]

fake_idx = np.argsort(coefs)[:10]
real_idx = np.argsort(coefs)[::-1][:10]

print('🔴 Top 10 Tokens Triggering FAKE Classification:')
for idx in fake_idx:
    print(f'  - {vocab[idx]:<25}: {coefs[idx]:.3f}')

print('\n🟢 Top 10 Tokens Triggering REAL Classification:')
for idx in real_idx:
    print(f'  - {vocab[idx]:<25}: +{coefs[idx]:.3f}')